# Word Embeddings : Word2Vec

TF-IDF treats every word as an unrelated column - it has no idea `cat` and `dog` are similar. Embeddings fix this by giving each word a vector where **similar words sit close together**.

In [ ]:
# !pip install gensim
# python < 3.14
# Tested on 3.13.12

In [ ]:
!python -c "import gensim; print(gensim.__version__)"

4.4.0


A tiny hand-made corpus built around the king/queen relationship - `* 3` repeats it so the model sees each pattern multiple times.

In [ ]:
# tiny corpus built around one specific relationship

sentences_raw = [
    # king/queen swapped in identical contexts -> forces the model to notice they play the same role
    "king is man",
    "queen is woman",
    "man is strong",
    "woman is strong",
    "king rules kingdom",
    "queen rules kingdom",
    "king wears crown",
    "queen wears crown",
    "prince is young man",
    "princess is young woman",
    "king is husband of queen",
    "queen is wife of king",
    "man works hard",
    "woman works hard",
    "king lives in palace",
    "queen lives in palace",
    "man walks to market",
    "woman walks to market",
    "boy grows into man",
    "girl grows into woman",
    "king is father of prince",
    "queen is mother of princess",
] * 3  # repeat the list 3x -> the model needs to see each pattern more than once to learn from it

In [ ]:
sentences_raw

['king is man',
 'queen is woman',
 'man is strong',
 'woman is strong',
 'king rules kingdom',
 'queen rules kingdom',
 'king wears crown',
 'queen wears crown',
 'prince is young man',
 'princess is young woman',
 'king is husband of queen',
 'queen is wife of king',
 'man works hard',
 'woman works hard',
 'king lives in palace',
 'queen lives in palace',
 'man walks to market',
 'woman walks to market',
 'boy grows into man',
 'girl grows into woman',
 'king is father of prince',
 'queen is mother of princess',
 'king is man',
 'queen is woman',
 'man is strong',
 'woman is strong',
 'king rules kingdom',
 'queen rules kingdom',
 'king wears crown',
 'queen wears crown',
 'prince is young man',
 'princess is young woman',
 'king is husband of queen',
 'queen is wife of king',
 'man works hard',
 'woman works hard',
 'king lives in palace',
 'queen lives in palace',
 'man walks to market',
 'woman walks to market',
 'boy grows into man',
 'girl grows into woman',
 'king is father of

Word2Vec expects a **list of token lists**, not raw strings - so we tokenize every sentence first.

In [ ]:
# split each sentence into words -> Word2Vec needs a list of token lists, not a list of strings
tokenized_sentences = [s.split() for s in sentences_raw]

In [ ]:
tokenized_sentences # Use stopword removal, lemmatization (POS(optional))

[['king', 'is', 'man'],
 ['queen', 'is', 'woman'],
 ['man', 'is', 'strong'],
 ['woman', 'is', 'strong'],
 ['king', 'rules', 'kingdom'],
 ['queen', 'rules', 'kingdom'],
 ['king', 'wears', 'crown'],
 ['queen', 'wears', 'crown'],
 ['prince', 'is', 'young', 'man'],
 ['princess', 'is', 'young', 'woman'],
 ['king', 'is', 'husband', 'of', 'queen'],
 ['queen', 'is', 'wife', 'of', 'king'],
 ['man', 'works', 'hard'],
 ['woman', 'works', 'hard'],
 ['king', 'lives', 'in', 'palace'],
 ['queen', 'lives', 'in', 'palace'],
 ['man', 'walks', 'to', 'market'],
 ['woman', 'walks', 'to', 'market'],
 ['boy', 'grows', 'into', 'man'],
 ['girl', 'grows', 'into', 'woman'],
 ['king', 'is', 'father', 'of', 'prince'],
 ['queen', 'is', 'mother', 'of', 'princess'],
 ['king', 'is', 'man'],
 ['queen', 'is', 'woman'],
 ['man', 'is', 'strong'],
 ['woman', 'is', 'strong'],
 ['king', 'rules', 'kingdom'],
 ['queen', 'rules', 'kingdom'],
 ['king', 'wears', 'crown'],
 ['queen', 'wears', 'crown'],
 ['prince', 'is', 'young', '

In [ ]:
print(f"Training on {len(tokenized_sentences)} sentences.")  # 22 sentences x 3 = 66
print("Example:", tokenized_sentences[0])                    # what one training sample looks like

### Train our own Word2Vec

The model learns a vector for every word by repeatedly trying to **predict a word's neighbours** - words appearing in similar contexts end up with similar vectors.

In [ ]:
from gensim.models import Word2Vec

In [ ]:
model = Word2Vec(
    sentences=tokenized_sentences,  # the training data : a list of token lists
    vector_size=20,  # how many numbers represent each word. real models use 100-300
    window=3,  # How many words left/right count as "context" for a given word
    sg=1,  # training algo, sg=1 <-- skip-gram, sg=0 <--- cbow. SKIP-GRAM <- predict content from word
    epochs=300,  # iteration <- how many passes over the data during training
    seed=1,  # fixes randomness so everyone in class gets the same vectors
    workers=0,  # Assignment --> watch batch 1.0 python advance recording --> thread and process
    min_count=1,  # ignore words appearing fewer that this many times in total.
)

In [ ]:
# model.wv = "word vectors". key_to_index maps each word -> its row in the vector matrix
print(f"Vocabulary size: {len(model.wv.key_to_index)}")

In [ ]:
# every unique word the model learned. only these 30 words have vectors
print(sorted(model.wv.key_to_index.keys()))

Every word is now a **dense vector of 20 numbers** (we set `vector_size=20`) - that vector *is* the embedding.

In [ ]:
king_vector = model.wv["king"]        # look up the learned vector for one word
print("Type:", type(king_vector))     # a numpy array, not a python list
print("Shape:", king_vector.shape)    # (20,) -> matches vector_size=20
print("Values:", king_vector)         # the 20 learned numbers. meaningless alone, useful when compared

`most_similar` finds the words whose vectors point in the most similar direction (cosine similarity, `1.0` = identical).

In [ ]:
print("Most similar to 'king':")
# topn=10 -> the 10 closest vectors. score is cosine similarity, 1.0 = identical direction
for word, score in model.wv.most_similar("king", topn=10):
    print(f"  {word:10s} {score:.4f}")  # :10s pads the word to 10 chars so the numbers line up

### The famous analogy test

Because words are vectors, you can do **arithmetic** on meaning : `king - man + woman` should land near `queen`.

In [ ]:
# king - man + woman --> ?
# positive = vectors to add, negative = vectors to subtract
result = model.wv.most_similar(positive=["king","woman"], negative=["man"],topn=5)

for word, score in result:
    print(f"  {word:10s} {score:.4f}")  # expected 'queen' at the top -> it is not there

The result is garbage - **66 tiny sentences is nowhere near enough data** to learn real word relationships. Embeddings need a massive corpus.

In [ ]:
# Why it failed:
# - 66 sentences / 30 words is far too small. Word2Vec needs millions of sentences
# - fix 1 : Need better data  -> load a model someone already trained on billions of words
# - fix 2 : poweful embedding models (Transformers) -> give a word a different vector per sentence

# Load Google Word2Vec model in python

Instead of training our own, load Google's - trained on **100 billion words** of Google News, giving 3 million words a **300-dimension** vector each (that's the `300` in the filename).

In [ ]:
# KeyedVectors = just the vectors, no training machinery. use it to load a pre-trained model
from gensim.models import KeyedVectors

In [ ]:
# binary=True because the .bin file stores raw floats, not text. takes a while, it is ~3.5 GB in memory
# model = KeyedVectors.load_word2vec_format('word2vec-GoogleNews-vectors/GoogleNews-vectors-negative300.bin.gz', binary=True)

In [ ]:
import gensim.downloader as api

model = api.load("word2vec-google-news-300")   # ~1.6 GB download, few minutes


In [ ]:
# same analogy as before. note: no .wv here, KeyedVectors *is* the vectors already
result = model.most_similar(positive=["king","woman"], negative=["man"],topn=1)

for word, score in result:
    print(f"  {word:10s} {score:.4f}")  # 'queen' -> same code, better data

Same analogy, real data → **`queen`**. Now let's check how well it understands similarity between everyday words.

In [ ]:
model.similarity('car', 'dog')  # unrelated things -> low score

In [ ]:
model.similarity('cat', 'dog')  # both pets, used in similar sentences -> high score

In [ ]:
# shares the letters "dog" but means food -> low score.
# proof the model learned from CONTEXT, not from spelling
model.similarity('hotdog', 'dog')

In [ ]:
# nobody told the model these are all vehicles. it worked that out purely from context
result = model.most_similar('car', topn=5)
result

### Limitation : out-of-vocabulary (OOV)

Word2Vec has a **fixed vocabulary** frozen at training time - a word it never saw (like `chatgpt`, which didn't exist in 2013) throws an error, so we catch it and return `-1`.

In [ ]:
def try_word_2_vec(word_1, word_2):
    try :
        return model.similarity(word_1, word_2)
    except:
        return -1  # word not in vocabulary -> KeyError. return -1 instead of crashing

In [ ]:
try_word_2_vec('chatgpt', 'cold')  # -1 : model trained in 2013, 'chatgpt' did not exist yet

In [ ]:
try_word_2_vec('weather', 'news')  # both in vocabulary -> real score comes back